# 🌐 AI Browser Agent (Qwen 2.5 Coder + Browser-Use)
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
Runs a **4-bit quantized LLM** (Qwen 2.5 Coder 7B) alongside a **headless Chromium browser** to create a fully autonomous web-browsing AI agent.
Designed for **Google Colab T4** and **Kaggle T4**.

## 🛠️ Step 1: Environment Setup & System Dependencies

In [ ]:
import os
try:
    import google.colab
    ENV = "Colab"
except ImportError:
    ENV = "Kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "Local"
print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Ensure 'Internet' is ON in Kaggle Notebook Settings.")

# System packages
!apt-get update -qq
!apt-get install -qq -y chromium-browser chromium-chromedriver xvfb

# Python packages
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q browser-use playwright
!python -m playwright install chromium

print("✅ Environment ready.")

## 🧠 Step 2: Load Quantized LLM (Qwen 2.5 Coder 7B)
4-bit NF4 quantization + `device_map='auto'` uses ~6 GB VRAM, leaving headroom for browsing tasks.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"⏳ Loading {MODEL_ID} in 4-bit mode...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()
print("✅ Model loaded across RAM/VRAM!")

if torch.cuda.is_available():
    u = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  🖥️  VRAM: {u:.2f} / {t:.2f} GB")

## 🌐 Step 3: Local LLM Inference Helper
A simple function that wraps the quantized model and generates text given a prompt.

In [ ]:
def llm_generate(prompt, max_new_tokens=512, temperature=0.2):
    """Generate text from the local quantized LLM."""
    messages = [
        {"role": "system",  "content": "You are an expert web automation assistant. Generate concise, correct Python code using Playwright to accomplish the user's task."},
        {"role": "user",    "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    out_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(out_ids, skip_special_tokens=True).strip()

# Smoke test
test_out = llm_generate("Write a one-line Playwright snippet to go to https://example.com")
print(f"Model test output:\n{test_out}")

## 🤖 Step 4: Browser-Use Agent Pipeline
The agent takes a natural language task, asks the LLM for a Playwright script, and executes it.

In [ ]:
import asyncio, re
from playwright.async_api import async_playwright

def extract_code(text):
    """Extract Python code block from LLM output."""
    match = re.search(r"```python\n(.*?)```", text, re.DOTALL)
    if match: return match.group(1).strip()
    match = re.search(r"```\n(.*?)```", text, re.DOTALL)
    if match: return match.group(1).strip()
    return text  # fallback: return raw output

async def run_browser_agent(task: str) -> str:
    """
    1. Ask LLM to write Playwright code for `task`.
    2. Execute the code in a headless Chromium browser.
    3. Return extracted page text.
    """
    print(f"\n🎯 Task: {task}")
    prompt = (
        f"Write async Playwright Python code to: {task}\n"
        "The code will be executed inside an existing `page` object. "
        "Do NOT include import statements or browser launch code. "
        "End by printing the result to stdout."
    )

    print("🤔 LLM thinking...")
    raw_code = llm_generate(prompt, max_new_tokens=512)
    code     = extract_code(raw_code)
    print(f"\n📝 Generated Code:\n{code}\n")

    result = "(no output)"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        page    = await browser.new_page()
        try:
            # Execute the generated code (with page in scope)
            exec_globals = {"page": page, "asyncio": asyncio}
            exec(f"async def _task(page):\n" + "\n".join(f"    {l}" for l in code.splitlines()), exec_globals)
            await exec_globals["_task"](page)
            result = await page.title()
            print(f"✅ Page title: {result}")
        except Exception as e:
            result = f"❌ Execution error: {e}"
            print(result)
        finally:
            await browser.close()
    return result

print("✅ Browser agent pipeline ready.")

## 🚀 Step 5: Run Agent Tasks
Edit the `tasks` list below and run the cell to let the agent browse the web for you!

In [ ]:
# ── Define tasks for the agent ─────────────────────────────────────────────────
tasks = [
    "Navigate to https://example.com and print the page heading.",
    "Go to https://httpbin.org/get and print the JSON response body.",
]

# ── Run all tasks sequentially ─────────────────────────────────────────────────
results = []
for task in tasks:
    r = await run_browser_agent(task)  # Jupyter supports top-level await
    results.append({"task": task, "result": r})

print("\n=== 📊 Results Summary ===")
for r in results:
    print(f"\n🎯 Task: {r['task']}")
    print(f"   Result: {r['result']}")

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*